# Implementing Multiple Turns

Now we write the **loop** that keeps calling Claude until it stops asking for tools — the piece `run_conversation` was missing last lesson.

**The signal:** `response.stop_reason`. When Claude wants a tool it's `"tool_use"`; anything else (`"end_turn"`) means it's done.

```python
if response.stop_reason != "tool_use":
    break   # Claude has a final answer
```

**Complete workflow:**
```
1. Send user message + tools
2. Claude replies with text and/or tool_use blocks
3. Run every requested tool → build tool_result blocks
4. Send results back as a user message
5. Repeat until stop_reason != "tool_use"
```

This lesson wires just **`get_current_datetime`** and shows *multi-turn* by asking for the time in two formats — Claude calls the tool twice, across two turns. Chaining *different* tools comes next lesson. Runs on Haiku 4.5.

## Setup — helpers, `chat`, `text_from_message` (from last lesson)

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from datetime import datetime
from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## The tool + schema

In [2]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

## Scalable tool routing: `run_tool`

A dispatch table mapping a tool **name** to its Python function. Adding a tool later means adding one branch — the conversation loop never changes. (Next lesson we extend this to `add_duration_to_datetime` and `set_reminder`.)

In [3]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    # elif tool_name == "another_tool":
    #     return another_tool(**tool_input)

## `run_tools` — execute every request, build result blocks

Filter the message for `tool_use` blocks and answer each with a matching `tool_result` block. Two important details:
- **`json.dumps(tool_output)`** — the result `content` must be a string; serializing keeps types unambiguous.
- **Error handling** — if a tool raises, we *still* return a `tool_result`, but with `is_error: True` and the error text. Claude can read that and potentially retry with corrected arguments (that's why clear error messages matter).

In [4]:
def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

## The conversation loop

Send → store the assistant turn → print any text → if it wasn't a tool request, stop → else run the tools, send the results back, and go again.

In [5]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

## Run it — one tool, two turns

Asking for two different formats makes Claude call `get_current_datetime` **twice**, across separate turns, before answering. Watch the loop iterate.

In [6]:
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?",
)

run_conversation(messages)


The current time is:
- **HH:MM format**: 23:23
- **SS format**: 12

So the current time is 23:23:12 (11:23:12 PM and 12 seconds).


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01FZGgKoACkDTZzhe9ABxx26', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01FBd8ymJACGVtyhDcLLKxJ7', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01FZGgKoACkDTZzhe9ABxx26',
    'content': '"23:23"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01FBd8ymJACGVtyhDcLLKxJ7',
    'content': '"12"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 23:23\n- **SS format**: 12\n\nSo the current time is 23:23:12 (11:23:12 PM and 12 seconds).', type='text')]}]

## Inspect the full history

The `messages` list tells the whole story: user question → assistant (text + tool_use) → user (tool_result) → assistant (tool_use again) → user (tool_result) → assistant (final text). Every block is preserved, which is what lets Claude build on earlier results.

In [7]:
messages

[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01FZGgKoACkDTZzhe9ABxx26', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01FBd8ymJACGVtyhDcLLKxJ7', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01FZGgKoACkDTZzhe9ABxx26',
    'content': '"23:23"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01FBd8ymJACGVtyhDcLLKxJ7',
    'content': '"12"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 23:23\n- **SS format**: 12\n\nSo the current time is 23:23:12 (11:23:12 PM and 12 seconds).', type='text')]}]

## Recap

The loop is tool-agnostic: `run_conversation` doesn't know or care which tools exist — it just checks `stop_reason`, delegates to `run_tools` → `run_tool`, and repeats. To add capability you touch only the **routing** and the **tools list**, never the loop.

Next: **Using multiple tools** — extend `run_tool` to all three reminder tools and pass every schema, so Claude can chain *different* tools (get now → add duration → set reminder) to fulfil "remind me in a week."